# 1. Load Fine-tuned Model

In [2]:
!pip install english_words

In [ ]:
print(english_words)

In [ ]:
# Define the path to the full fine-tuned model
MODEL_PATH = "/content/drive/MyDrive/spelling_correction_project/indot5_spell_checker_v4_fft.zip"

# Extract the model files from the zip archive
import zipfile
with zipfile.ZipFile(MODEL_PATH, 'r') as zip_ref:
    zip_ref.extractall('/content/model_t5_fft')

# Load the model and tokenizer using Hugging Face's Transformers pipeline
from transformers import pipeline

In [ ]:
t5_model_fft = "/content/model_t5_fft/indot5_spell_checker_v4_fft"
corrector = pipeline(
    "text2text-generation",
    model=t5_model_fft,
    tokenizer=t5_model_fft
)

Device set to use cpu


In [ ]:
# Test correction with a sample sentence
input_sentence = input(str("masukkan kalimat = "))

corrected_sentence = corrector(input_sentence, max_length=512)[0]["generated_text"]
print("Original:", input_sentence)
print("Corrected:", corrected_sentence)

masukkan kalimat = aku tdka sabar menungguny
Original: aku tdka sabar menungguny
Corrected: aku tidak sabar menunggunya.


In [ ]:
input_sentence = "bagaimaa bunyiya"
corrected_sentence = corrector(input_sentence, max_length=128)[0]["generated_text"]
print("Original:", input_sentence)
print("Corrected:", corrected_sentence)

Original: bagaimaa bunyiya
Corrected: bagaimana bunyiya?


# 2. Deployment using Streamlit

In [ ]:
!pip install transformers streamlit torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.3 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import time

# Set page title
st.set_page_config(page_title="Indonesian Spelling Error Correction")

# Define the path to the full fine-tuned model
MODEL_PATH = "/content/drive/MyDrive/spelling_correction_project/indot5_spell_checker_v4_fft.zip"

# Extract the model files from the zip archive
import zipfile
with zipfile.ZipFile(MODEL_PATH, 'r') as zip_ref:
    zip_ref.extractall('/content/model_t5_fft')


# Load model and tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Load the model (with caching to improve performance)
@st.cache_resource()
def load_model():
    corrector = pipeline(
        "text2text-generation",
        model=t5_model_fft,
        tokenizer=t5_model_fft
        )
    return model

corrector = load_model()

# Streamlit interface
st.title("Indonesian Spelling Error Correction with IndoT5")
st.header("Enter text for correction:")

input_text = st.text_area("Write anything:", height=200)

# Upload a file
file_upload = st.file_uploader("Choose a file", type=['txt', 'docx', 'pdf'])
if file_upload is not None:
    text = file_upload.read().decode("utf-8")
    st.text_area("Document Content:", value=text, height=200)

    # Run prediction
    if st.button("Predict and Correct"):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        outputs = model.generate(inputs["input_ids"])
        corrected_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        st.subheader("Suggestions:")
        st.write(corrected_text)


Writing app.py


In [ ]:
%%writefile app.py

import streamlit as st
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import zipfile
import os

# Set paths
MODEL_PATH = "models/spell-checker-v6.zip"
EXTRACT_DIR = "model_dir"

# Extract model once
if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(MODEL_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

# Load model
@st.cache_resource
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(EXTRACT_DIR)
    model = AutoModelForSeq2SeqLM.from_pretrained(EXTRACT_DIR)
    return pipeline("text2text-generation", model=model, tokenizer=tokenizer)

# Streamlit app
st.title("Indonesian Spell Correction App")
corrector = load_model()

# File uploader
uploaded_file = st.file_uploader("Choose a file", type=["txt", "docx", "pdf"])
if uploaded_file:
    st.write("Processing...")
    content = uploaded_file.read().decode("utf-8")
    corrected = corrector(content, truncation=True, max_length=512)
    st.text_area("Corrected Text", corrected[0]['generated_text'], height=300)

Overwriting app.py


In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 8s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [ ]:
# public ip is the password to the localtunnel
!curl ipv4.icanhazip.com

35.193.255.108


In [ ]:
!streamlit run app.py &>./logs.txt & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧your url is: https://afraid-mammals-stand.loca.lt


# 3. Error Highlighting

In [ ]:
import streamlit as st
from difflib import SequenceMatcher
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer
MODEL_PATH = "./model_t5_save"
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

st.title("Indonesian Spelling Error Correction")
st.write("Unggah dokumenmu untuk memperbaiki kesalahan ejaan")

# File upload
uploaded_file = st.file_uploader("Choose a file", type=["txt", "docx", "pdf"])
if uploaded_file is not None:
    text = uploaded_file.read().decode("utf-8")
    st.text_area("Konten Original:", value=text, height=200)

    if st.button("Analisa dan Perbaiki"):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        outputs = model.generate(inputs["input_ids"])
        corrected_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        st.subheader("Teks yang Dikoreksi:")
        st.text_area("Konten Hasil:", value=corrected_text, height=200)

        # Highlight differences
        def highlight_differences(original, corrected):
            matcher = SequenceMatcher(None, original, corrected)
            highlighted = []
            for tag, i1, i2, j1, j2 in matcher.get_opcodes():
                if tag == "equal":
                    highlighted.append(original[i1:i2])
                else:
                    highlighted.append(f"**[ERROR: {original[i1:i2]} -> {corrected[j1:j2]}]**")
            return " ".join(highlighted)

        highlighted_text = highlight_differences(text, corrected_text)
        st.markdown("### Highlighted Errors:")
        st.markdown(highlighted_text, unsafe_allow_html=True)

        # Suggestions (Mockup)
        st.subheader("Correction Suggestions:")
        suggestions_df = pd.DataFrame({
            "Error": ["iut", "skses"],
            "Suggestion 1 (Prob: 90%)": ["itu", "sukses"],
            "Suggestion 2 (Prob: 80%)": ["gitu", "fail"],
        })
        st.write(suggestions_df)

        # Option to download corrected file
        corrected_filename = "corrected_document.txt"
        with open(corrected_filename, "w") as f:
            f.write(corrected_text)
        st.download_button("Download Corrected File", data=corrected_text, file_name="corrected_document.txt", mime="text/plain")
        st.success("Analisa selesai!")

# 4. Testing

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(t5_model_fft)
model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_fft)

# Test input
input_sentence = "merupkan sebuh proses"

# Generate corrected output
inputs = tokenizer(input_sentence, max_length=512, return_tensors="pt", truncation=True)
outputs = model.generate(inputs["input_ids"], max_length=512, num_beams=5, early_stopping=True)
corrected_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Original:", input_sentence)
print("Corrected:", corrected_sentence)

Original: merupkan sebuh proses
Corrected: merupakan sebuah proses.
